### Exercises

#### Exercise 1

Check to see if any of the players are duplicated (appear more than one time in the dataset).

In [1]:
import pandas as pd

df = pd.read_csv("../data/fifa.csv", low_memory=False)

duplicate_names = df[df["LongName"].duplicated()]["LongName"]

print(duplicate_names)

944       Kevin Berlaso
1187     Lisandro López
2516     Gonzalo Castro
2562         Ben Davies
2955      Claudio Bravo
              ...      
18703        Yuan Zhang
18741       Alfie Jones
18813        Joe Wright
18834        Mark Byrne
18909         Wei Zhang
Name: LongName, Length: 128, dtype: object


#### Exercise 2

Inspect the dataset to see if there are any problems with missing data. If there are, determine how you would like to handle these missing values and then proceed to implement your chosen method.

In [8]:
# Drop the urls as they produce images which are not needed here
df_labelsdrop = df.drop(labels = ["photoUrl", "playerUrl"], axis = 1)
df_labelsdrop.info()
# Outside of url blanks, it is viable to drop the rest due to the size of the dataset compared to the blank cells
df_deleted = df.dropna()
df_deleted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18979 entries, 0 to 18978
Data columns (total 75 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   LongName          18979 non-null  object
 1   Nationality       18979 non-null  object
 2   Positions         18979 non-null  object
 3   Name              18979 non-null  object
 4   Age               18979 non-null  int64 
 5   ↓OVA              18979 non-null  int64 
 6   POT               18979 non-null  int64 
 7   Team & Contract   18979 non-null  object
 8   ID                18979 non-null  int64 
 9   Height            18979 non-null  object
 10  Weight            18979 non-null  object
 11  foot              18979 non-null  object
 12  BOV               18979 non-null  int64 
 13  BP                18979 non-null  object
 14  Growth            18979 non-null  int64 
 15  Joined            18979 non-null  object
 16  Loan Date End     1013 non-null   object
 17  Value       

#### Exercise 3

Use mean imputation to replace missing values in the Hits column.

In [6]:
k_to_1000 = 1000

hits_multiplier = (
    df["Hits"].str.contains("K")
    .map(lambda x: k_to_1000 if x else 1)
)

df["Hits"] = (
    df["Hits"].str.replace("K", "")
    .astype("float")
    *hits_multiplier
)

df["Hits"] = df["Hits"].fillna(df["Hits"].mean())

#### Exercise 4

Remove unwanted text from the Weight column.

In [28]:
df["Weight"].unique()
df["Weight"] = df["Weight"].str.replace("lbs", "")

AttributeError: Can only use .str accessor with string values!

#### Exercise 5

Remove unwanted text from the Value and Wage columns.

In [11]:
value_multiplier = (
    df["Value"].str.contains("M")
    .map(lambda x: 1000000 if x else 1000)
)

df["Value"] = (
    df["Value"].str.replace("€", "")
    .str.replace("M", "")
    .str.replace("K", "")
    .astype("float")*value_multiplier
)
df["Wage"] = (
    df["Wage"].str.replace("€", "")
    .str.replace("M", "")
    .str.replace("K", "")
    .astype("float")*value_multiplier
)

#### Exercise 6

Convert the Weight, Value, and Wage columns to the `float` data type.

In [12]:
df["Weight"] = df["Weight"].astype("float")
df["Value"] = df["Value"].astype("float")
df["Wage"] = df["Wage"].astype("float")

#### Exercise 7

Utilize method chaining to do the following in a single pass:

1. Remove unwanted characters from the Release Clause column.
2. Convert data type to `float`
3. Impute missing values using median

In [27]:
df["Release Clause"].unique()
(df["Release Clause"].str.replace("€", "")
    .str.replace("M", "")
    .str.replace("K", "")
    .astype("float")
    .pipe(lambda x: x.fillna(x.median()))
)

array(['€138.4M', '€75.9M', '€159.4M', ..., '€101K', '€64K', '€35K'],
      dtype=object)

#### Exercise 8

In the Height column, determine which observations are in centimetres and which are in inches. Save the results in another variable. Then use method chaining to do the following in a single pass:

1. Remove unwanted characters from the Height column.
2. Convert data type to `float`
3. Use the previously computed variable with the inches/centimetres indicator to convert all values to inches.

In [32]:
cm_to_in = 0.393701

height_multiplier = (
    df["Height"].str.contains(" centimetres")
    .map(lambda x: cm_to_in if x else 1)
)
df["Height"] = (
    df["Height"].str.replace(" centimetres", "")
    .str.replace(" inches", "")
    .astype("float") * height_multiplier
)
df["Height"].unique()

array([67.       , 74.015788 , 71.       , 69.0157853, 72.       ,
       70.0000378, 75.0000405, 69.       , 75.984293 , 74.       ,
       73.       , 78.       , 76.       , 72.9921654, 67.9921627,
       65.9842876, 72.0079129, 70.9842903, 70.       , 77.0079156,
       68.       , 65.       , 75.       , 66.       , 77.       ,
       64.0157826, 67.0079102, 77.9921681, 79.       , 65.0000351,
       64.       , 79.0157907, 62.       , 62.99216  , 63.       ,
       80.0000432, 60.9842849, 80.       , 81.       ])

#### Exercise 9

Fix the foot column so that all values are either Left or Right.

In [25]:
df["foot"].unique()

foot_dict = {}
foot_dict["Left"] = "Left"
foot_dict["left"] = "Left"
foot_dict["L"] = "Left"
foot_dict["Right"] = "Right"
foot_dict["right"] = "Right"
foot_dict["R"] = "Right"
df["foot"] = df["foot"].map(foot_dict)
df["foot"].unique()

array(['Left', 'Right'], dtype=object)

#### Exercise 10

Check the Wage column for any outliers and then output those observations marked as outliers.

In [29]:
Q1 = df["Wage"].quantile(0.25)
Q3 = df["Wage"].quantile(0.75)
IQR = Q3-Q1

df[(df["Wage"] < (Q1 - 1.5*IQR)) | (df["Wage"] > (Q3 + 1.5*IQR))].head()

,photoUrl,LongName,playerUrl,Nationality,Positions,Name,Age,↓OVA,POT,Team & Contract,...,A/W,D/W,IR,PAC,SHO,PAS,DRI,DEF,PHY,Hits
0,https://cdn.sofifa.com/players/158/023/21_60.png,Lionel Messi,http://sofifa.com/player/158023/lionel-messi/2...,Argentina,RW ST CF,L. Messi,33,93,93,\n\n\n\nFC Barcelona\n2004 ~ 2021\n\n,...,Medium,Low,5 ★,85,92,91,95,38,65,372.0
1,https://cdn.sofifa.com/players/020/801/21_60.png,C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-s...,Portugal,ST LW,Cristiano Ronaldo,35,92,92,\n\n\n\nJuventus\n2018 ~ 2022\n\n,...,High,Low,5 ★,89,93,81,89,35,77,344.0
2,https://cdn.sofifa.com/players/200/389/21_60.png,Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,GK,J. Oblak,27,91,93,\n\n\n\nAtlético Madrid\n2014 ~ 2023\n\n,...,Medium,Medium,3 ★,87,92,78,90,52,90,86.0
3,https://cdn.sofifa.com/players/192/985/21_60.png,Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyn...,Belgium,CAM CM,K. De Bruyne,29,91,91,\n\n\n\nManchester City\n2015 ~ 2023\n\n,...,High,High,4 ★,76,86,93,88,64,78,163.0
4,https://cdn.sofifa.com/players/190/871/21_60.png,Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silv...,Brazil,LW CAM,Neymar Jr,28,91,91,\n\n\n\nParis Saint-Germain\n2017 ~ 2022\n\n,...,High,Medium,5 ★,91,85,86,94,36,59,273.0
